In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

env = 'local'
# env = 'colab'

In [3]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

# 载入final_oos_df
if env == 'local':
    final_oos_df = pd.read_parquet('/Users/jianbinchen/Downloads/final_oos_df_20260718_1535.parquet') 
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'
elif env == 'colab':
    import subprocess
    from google.colab import drive
    drive.mount("/content/drive")
    path = '/content/drive/MyDrive/Colab Notebooks/'

    def read_drive_parquet(link):
        file_id = link.split("/d/")[1].split("/")[0]
        path = "/content/final_oos_df.parquet"
        url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
        subprocess.run(["wget", "-q", "-O", path, url], check=True)
        return pd.read_parquet(path)

    final_oos_df = read_drive_parquet(
        "https://drive.google.com/file/d/1sN3mceX_Wtnxvv24BIefilfXWqwhzX9L/view?usp=drive_link")

display(pd.concat([final_oos_df.head(5), final_oos_df.tail(5)], ignore_index=True))

,permno,mthcaldt,target_ret_final,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2006-01-31,-0.010537,"[0.014396755024790764, 0.015741733834147453, 0...","[-2.1292262077331543, -2.0399136543273926, 1.7...","[-0.02683735452592373, 0.002639785408973694, 0...","[0.060575105249881744, 0.05262312665581703, 0....","[4.562270641326904, 3.745076894760132, 20.1312..."
1,10002,2006-01-31,0.065701,"[0.014396755024790764, 0.015741733834147453, 0...","[-2.1292262077331543, -2.0399136543273926, 1.7...","[-0.015064217150211334, 0.018360594287514687, ...","[0.05121513828635216, 0.05009158328175545, 0.0...","[5.687742233276367, 6.253466606140137, 38.2552..."
2,10025,2006-01-31,0.035385,"[0.014396755024790764, 0.015741733834147453, 0...","[-2.1292262077331543, -2.0399136543273926, 1.7...","[-0.0751136988401413, -0.03647824376821518, 0....","[0.0872146487236023, 0.06643757224082947, 0.06...","[5.31073522567749, 3.7359538078308105, 10.1180..."
3,10026,2006-01-31,0.121202,"[0.014396755024790764, 0.015741733834147453, 0...","[-2.1292262077331543, -2.0399136543273926, 1.7...","[-0.05124913901090622, -0.012725161388516426, ...","[0.06258069723844528, 0.05291612818837166, 0.0...","[5.523638725280762, 4.593089580535889, 19.4625..."
4,10032,2006-01-31,0.185447,"[0.014396755024790764, 0.015741733834147453, 0...","[-2.1292262077331543, -2.0399136543273926, 1.7...","[-0.13419784605503082, -0.09679749608039856, -...","[0.10556411743164062, 0.08268610388040543, 0.0...","[7.601531028747559, 6.252680778503418, 8.23628..."
5,93374,2024-12-31,0.012492,"[8.926969030653709e-07, 0.01750946417450905, 0...","[-10.596389770507812, -0.7123852968215942, 3.0...","[-0.042454205453395844, 0.037027306854724884, ...","[0.05423310399055481, 0.05205207318067551, 0.0...","[3.1335577964782715, 8.962891578674316, 23.924..."
6,93397,2024-12-31,-0.057236,"[8.926969030653709e-07, 0.01750946417450905, 0...","[-10.596389770507812, -0.7123852968215942, 3.0...","[-0.014532210305333138, 0.02865726128220558, 0...","[0.09144271910190582, 0.07327396422624588, 0.0...","[5.828946113586426, 4.5879693031311035, 8.6893..."
7,93426,2024-12-31,-0.007243,"[8.926969030653709e-07, 0.01750946417450905, 0...","[-10.596389770507812, -0.7123852968215942, 3.0...","[0.011092176660895348, 0.053119637072086334, 0...","[0.11711584031581879, 0.09086589515209198, 0.0...","[6.8036298751831055, 5.016971588134766, 7.0060..."
8,93434,2024-12-31,0.147685,"[8.926969030653709e-07, 0.01750946417450905, 0...","[-10.596389770507812, -0.7123852968215942, 3.0...","[-0.020737765356898308, 0.021697310730814934, ...","[0.2869930863380432, 0.20450833439826965, 0.20...","[17.079084396362305, 6.449190139770508, 10.071..."
9,93436,2024-12-31,0.001882,"[8.926969030653709e-07, 0.01750946417450905, 0...","[-10.596389770507812, -0.7123852968215942, 3.0...","[-0.015368333086371422, 0.024617811664938927, ...","[0.13341635465621948, 0.10031997412443161, 0.0...","[8.394401550292969, 8.286294937133789, 10.5064..."


In [4]:
# =================================================================
# 1. 核心数学工具：混合高斯分布的 CDF 和 ES 计算
# =================================================================

def mdn_cdf(y, pi, mu, sigma, nu=None):
    """
    计算混合高斯分布在 y 处的累积概率 P(Y <= y)
    公式: F(y) = sum( pi_k * Phi((y - mu_k) / sigma_k) )
    """
    """
    兼容 N 和 t 的混合分布累积概率
    """
    if nu is None:
        # norm.cdf 是标准正态分布的累积函数
        return np.sum(pi * norm.cdf((y - mu) / sigma))
    else:
        # scipy.stats.t.cdf 的 loc(均值) 默认0，scale(标准差) 默认1，需要标准化 y
        return np.sum(pi * t.cdf((y - mu) / sigma, df=nu))


def mdn_var_objective(y, pi, mu, sigma, nu, alpha):
    """
    求解 VaR 的目标函数：F(y) - alpha = 0
    """
    return mdn_cdf(y, pi, mu, sigma, nu) - alpha

def calculate_es_analytical(alpha, pi, mu, sigma, nu=None, right_tail=False):
    """
    right_tail=False: 左尾 ES，即 E[X | X ≤ VaR_α]，用于 alpha=1%/5%/10%
    right_tail=True:  右尾 ES，即 E[X | X ≥ VaR_α]，用于 alpha=90%/95%/99%
    """
    # ==========================================
    # VaR 求根部分完全不变
    # ==========================================
    lower_bound = np.min(mu) - 10 * np.max(sigma)
    upper_bound = np.max(mu) + 10 * np.max(sigma)
    step = np.max(sigma) * 5 + 0.1

    for _ in range(100):
        val_lower = mdn_var_objective(lower_bound, pi, mu, sigma, nu, alpha)
        val_upper = mdn_var_objective(upper_bound, pi, mu, sigma, nu, alpha)
        if val_lower < 0 and val_upper > 0:
            break
        if val_lower >= 0:
            lower_bound -= step
        if val_upper <= 0:
            upper_bound += step
        step *= 1.5

    try:
        var_alpha = brentq(mdn_var_objective, lower_bound, upper_bound, args=(pi, mu, sigma, nu, alpha))
    except Exception as e:
        print(f">>> Warning: Root finding failed safely. {e}")
        return np.sum(pi * mu), np.sum(pi * mu)

    # ==========================================
    # ES 积分：根据 right_tail 切换公式
    # ==========================================
    d = (var_alpha - mu) / sigma

    if not right_tail:
        # 左尾：E[X | X ≤ VaR]，对应 alpha = 1%/5%/10%
        if nu is None:
            term1 = mu * norm.cdf(d)
            term2 = sigma * norm.pdf(d)
        else:
            term1 = mu * t.cdf(d, df=nu)
            term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
        es_val = -(1 / alpha) * np.sum(pi * (term1 - term2))

    else:
        # 右尾：E[X | X ≥ VaR]，对应 alpha = 90%/95%/99%
        # 公式：(1/(1-α)) * Σ pi_k * [mu_k*(1-F(d)) + sigma_k*f(d)*调整项]
        if nu is None:
            term1 = mu * (1 - norm.cdf(d))
            term2 = sigma * norm.pdf(d)
        else:
            term1 = mu * (1 - t.cdf(d, df=nu))
            term2 = sigma * ((nu + d**2) / (nu - 1)) * t.pdf(d, df=nu)
        es_val = (1 / (1 - alpha)) * np.sum(pi * (term1 + term2))

    return var_alpha, es_val

def calculate_crps_numerical(y_true, pi, mu, sigma, nu=None):
    """
    通过数值积分计算混合高斯分布的 CRPS
    """
    # 1. 构建积分网格 z
    # 因为我们的目标收益率已经用 VIX 缩放过，绝大多数值落在 [-5, 5] 之间。
    # 设置 [-10, 10] 的区间和 1000 个格点，足以保证极高的积分精度。
    z = np.linspace(-10, 10, 1000)
    
    # 2. 计算网格上每一个点的预测累积概率 F(z)
    cdf_z = np.zeros_like(z)
    for k in range(len(pi)):
        # 向量化计算：算出每一个网格点的正态 CDF 并加权
        if nu is None:
            cdf_z += pi[k] * norm.cdf((z - mu[k]) / sigma[k])
        else:
            cdf_z += pi[k] * t.cdf((z - mu[k]) / sigma[k], df=nu[k])

    # 3. 真实发生的阶跃函数 (Heaviside step function)
    # 当 z 小于真实收益率时为 0，大于等于时为 1
    step_z = (z >= y_true).astype(float)
    
    # 4. 计算平方差
    squared_diff = (cdf_z - step_z) ** 2
    
    # 5. 使用梯形法则 (Trapezoidal rule) 计算定积分
    crps_val = np.trapezoid(squared_diff, z)
    
    return crps_val



def calculate_tail_result(alpha, pi, mu, sigma, nu=None):
    right_tail = alpha > 0.5
    var_value, es_value = calculate_es_analytical(
        alpha, pi, mu, sigma, nu, right_tail=right_tail
    )
    alpha_label = f"{alpha * 100:g}".replace('.', '_')
    return {
        f'es_{alpha_label}': es_value,
        f'var_{alpha_label}': var_value,
    }

def calculate_pit_result(y_true, pi, mu, sigma, nu=None):
    return {'pit': mdn_cdf(y_true, pi, mu, sigma, nu)}


def calculate_crps_result(y_true, pi, mu, sigma, nu=None):
    return {'crps': calculate_crps_numerical(y_true, pi, mu, sigma, nu)}


In [5]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_test = "quick"  # "quick" or "full"
alphas = [0.01, 0.05]

# =================================================================
# 2. 主程序：从 SQLite 读取数据并执行计算
# =================================================================
results = []

total = len(final_oos_df)
print_every = max(1, total // 20)  # 每 5% 打印一次

print(">>> Extracting Tail Risk Factors (ES & VaR)...")
for idx, (df_idx, row) in enumerate(final_oos_df.iterrows()):
    if run_test == "quick" and idx >= 100:  # 测试模式下只跑前100行
        break
    # 每 x% 打印一次进度
    if idx % print_every == 0:
        print(f"  Progress: {idx/total*100:.0f}% ({idx}/{total})")
    # 1. 解码 JSON 字符串回 Numpy 数组
    pi = np.array(json.loads(row['pi_vec']))
    mu = np.array(json.loads(row['mu_vec']))
    sigma = np.array(json.loads(row['sigma_vec']))
    y_true = row['target_ret_final'] # 真实发生的（缩放后的）收益率

    # 🌟 2. 安全读取自由度 nu (智能判断分布类型)
    nu = None
    if 'nu_vec' in row and pd.notna(row['nu_vec']):
        nu = np.array(json.loads(row['nu_vec']))

    tail_result = {}
    for alpha in alphas:
        tail_result.update(
            calculate_tail_result(alpha, pi, mu, sigma, nu)
        )
    pit_result = calculate_pit_result(y_true, pi, mu, sigma, nu)
    crps_result = calculate_crps_result(y_true, pi, mu, sigma, nu)

    # 🌟 3. 高阶矩计算 (方差会随 t 分布自由度变化)
    mean_pred = np.sum(pi * mu)
    if nu is None:
        component_var = sigma**2
    else:
        # t分布的真实方差公式: sigma^2 * (nu / (nu - 2))
        # (我们的网络中 nu_head 加了 2.01，所以 nu 必然大于 2，不会出现除以0的错误)
        component_var = sigma**2 * (nu / (nu - 2))
    # 混合分布方差: sum(pi * (mu^2 + sigma^2)) - mean^2
    var_pred = np.sum(pi * (mu**2 + component_var)) - mean_pred**2

    # 将 VaR/ES、PIT 和 CRPS 合并进当前结果
    results.append({
        'permno': row['permno'],
        'mthcaldt': row['mthcaldt'],
        **tail_result,
        **pit_result,
        **crps_result,
        'mean_pred': mean_pred,
        'vol_pred': np.sqrt(var_pred),
        'realized_ret': y_true
    })

ES_df = pd.DataFrame(results)

>>> Extracting Tail Risk Factors (ES & VaR)...
  Progress: 0% (0/731890)


In [6]:
ES_df.to_parquet(os.path.join(path, f'ES_df_{timestamp}.parquet'), index=False)
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output


In [7]:
ES_df.head(5)

,permno,mthcaldt,es_1,var_1,es_5,var_5,pit,crps,mean_pred,vol_pred,realized_ret
0,10001,2006-01-31,0.164824,-0.114921,0.101694,-0.068546,0.259546,0.027589,0.030865,0.077262,-0.010537
1,10002,2006-01-31,0.157261,-0.124221,0.111327,-0.084742,0.881335,0.035395,0.003047,0.058005,0.065701
2,10025,2006-01-31,0.236519,-0.171952,0.149750,-0.102788,0.582665,0.019799,0.026994,0.094456,0.035385
3,10026,2006-01-31,0.167104,-0.125900,0.110475,-0.078594,0.938683,0.070139,0.020672,0.069517,0.121202
4,10032,2006-01-31,0.313440,-0.251492,0.219273,-0.163080,0.953162,0.128665,0.000174,0.107880,0.185447


In [ ]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()